# Decision Trees and Random Forest Project  
## Bank Marketing Dataset

### Project Goal
In this project, we will use the **Bank Marketing Dataset** to predict whether a customer will subscribe to a term deposit or not.

The target column is:

`y`

Where:

- `yes` means the customer subscribed.
- `no` means the customer did not subscribe.

We will build and compare two classification models:

1. **Decision Tree Classifier**
2. **Random Forest Classifier**

# Import Libraries

Import the usual libraries for data analysis, visualization, and machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

# Get the Data

Use pandas to read the dataset as a dataframe called `bank`.

Important note:  
This dataset is separated by semicolons `;`, so we must use:

`sep=';'`

In [ ]:
bank = pd.read_csv('bank.csv', sep=';')

Check the first rows of the dataset.

In [ ]:
bank.head()

Check the dataset information.

In [ ]:
bank.info()

Check the statistical summary of the numerical columns.

In [ ]:
bank.describe()

Check the shape of the dataset.

In [ ]:
bank.shape

Check if there are missing values.

In [ ]:
bank.isnull().sum()

Check the target column distribution.

In [ ]:
bank['y'].value_counts()

In [ ]:
sns.countplot(x='y', data=bank)
plt.title('Target Distribution: Subscribed to Term Deposit')
plt.xlabel('Subscribed')
plt.ylabel('Count')
plt.show()

# Exploratory Data Analysis

Let's do some data visualization to understand the dataset.

We will create similar plots to the original project, but using suitable columns from the Bank Marketing Dataset.

## 1. Histogram of Age by Housing Loan

Create a histogram showing the age distribution for customers who have a housing loan and customers who do not.

In [ ]:
plt.figure(figsize=(10,6))
bank[bank['housing']=='yes']['age'].hist(alpha=0.5, bins=30, label='housing = yes')
bank[bank['housing']=='no']['age'].hist(alpha=0.5, bins=30, label='housing = no')
plt.legend()
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.title('Age Distribution by Housing Loan')
plt.show()

## 2. Histogram of Age by Subscription Status

Create a similar histogram, but this time based on the target column `y`.

In [ ]:
plt.figure(figsize=(10,6))
bank[bank['y']=='yes']['age'].hist(alpha=0.5, bins=30, label='Subscribed = yes')
bank[bank['y']=='no']['age'].hist(alpha=0.5, bins=30, label='Subscribed = no')
plt.legend()
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.title('Age Distribution by Subscription Status')
plt.show()

## 3. Countplot of Job by Subscription Status

Create a countplot showing the counts of customers by job type, with the color hue defined by the target column `y`.

In [ ]:
plt.figure(figsize=(14,6))
sns.countplot(x='job', data=bank, hue='y')
plt.xticks(rotation=45)
plt.title('Job Type Count by Subscription Status')
plt.xlabel('Job')
plt.ylabel('Count')
plt.show()

## 4. Jointplot Between Duration and Balance

Let's see the relationship between call duration and customer balance.

In [ ]:
sns.jointplot(x='duration', y='balance', data=bank)
plt.show()

## 5. lmplot to Compare Trends

Create an lmplot to see whether the relationship between age and balance differs based on subscription status and housing loan status.

In [ ]:
sns.lmplot(x='age', y='balance', data=bank, hue='y', col='housing', height=5, aspect=1)
plt.show()

## 6. Correlation Heatmap for Numerical Features

This plot helps us see the relationships between numerical variables.

In [ ]:
plt.figure(figsize=(10,6))
numeric_bank = bank.select_dtypes(include=['int64', 'float64'])
sns.heatmap(numeric_bank.corr(), annot=True, cmap='viridis')
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

# Setting up the Data

Now we will prepare the data for our classification models.

Machine learning models from scikit-learn cannot directly understand text categories such as `job`, `marital`, or `education`.

Therefore, we need to convert categorical columns into dummy variables using:

`pd.get_dummies()`

Check the dataset information again.

In [ ]:
bank.info()

## Categorical Features

These are the categorical columns that need to be converted into dummy variables.

In [ ]:
cat_feats = ['job', 'marital', 'education', 'default', 'housing',
             'loan', 'contact', 'month', 'poutcome']

Before applying dummy variables, convert the target column `y` into numbers:

- `yes` becomes `1`
- `no` becomes `0`

In [ ]:
bank_model = bank.copy()
bank_model['y'] = bank_model['y'].map({'no': 0, 'yes': 1})

Now use `pd.get_dummies()` to create a final dataframe with dummy variables.

In [ ]:
final_data = pd.get_dummies(bank_model, columns=cat_feats, drop_first=True)

In [ ]:
final_data.head()

In [ ]:
final_data.info()

# Train Test Split

Now it is time to split the data into a training set and a testing set.

We will use:

- 70% for training
- 30% for testing

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = final_data.drop('y', axis=1)
y = final_data['y']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=101
)

# Training a Decision Tree Model

First, we will train a single Decision Tree model.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
dtree = DecisionTreeClassifier(random_state=101)

In [ ]:
dtree.fit(X_train, y_train)

# Predictions and Evaluation for Decision Tree

Now evaluate the Decision Tree model using:

- Confusion Matrix
- Classification Report

In [ ]:
predictions = dtree.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
print(confusion_matrix(y_test, predictions))

In [ ]:
print(classification_report(y_test, predictions))

# Training the Random Forest Model

Now we will train a Random Forest model and compare it with the Decision Tree model.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rfc = RandomForestClassifier(n_estimators=100, random_state=101)

In [ ]:
rfc.fit(X_train, y_train)

# Predictions and Evaluation for Random Forest

In [ ]:
rfc_pred = rfc.predict(X_test)

In [ ]:
print(confusion_matrix(y_test, rfc_pred))

In [ ]:
print(classification_report(y_test, rfc_pred))

# Final Model Comparison

Now we compare the two models using common classification metrics:

- Accuracy
- Precision
- Recall
- F1-score

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

comparison = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, predictions),
        accuracy_score(y_test, rfc_pred)
    ],
    'Precision': [
        precision_score(y_test, predictions),
        precision_score(y_test, rfc_pred)
    ],
    'Recall': [
        recall_score(y_test, predictions),
        recall_score(y_test, rfc_pred)
    ],
    'F1-Score': [
        f1_score(y_test, predictions),
        f1_score(y_test, rfc_pred)
    ]
})

comparison

# Conclusion

In this project, we used the Bank Marketing Dataset to predict whether a customer will subscribe to a term deposit or not.

We trained two models:

1. Decision Tree Classifier
2. Random Forest Classifier

After comparing the models using the classification report and the comparison table, we can decide which model performed better.

Usually, Random Forest performs better than a single Decision Tree because it combines multiple trees and reduces overfitting.

# Notes for Submission

Make sure the following files are in the same folder:

1. This notebook file
2. `bank.csv`

Then run the notebook cells from top to bottom.